# DWthon Memory - Dzień 3: `AI Language Coach, który nie zaczyna od zera`

Dziś dowozimy publiczną obietnicę wydarzenia w najbardziej dosłownej formie: budujemy **AI Language Coach**, który nie tylko odpowiada, ale też **pamięta użytkownika, kontekst i postęp między sesjami**. Wczoraj zbudowaliśmy silnik pamięci. Dziś opakowujemy go w **produkt MVP, który można pokazać klientowi, zespołowi albo inwestorowi** i od razu poczuć, że to nie jest zwykły chat.

Do `Core` dokładamy też bardzo ważną zasadę: **`CONTINUITY`**. Jeśli pamięć daje kontekst, Coach ma z niego skorzystać. To właśnie ona pilnuje, żeby nowa sesja nie startowała od zera, gdy system już coś wie.


## Czego nauczysz się dzisiaj?

1. Zobaczysz, jak przejść od pojedynczego `state` do **store wielu użytkowników**.
2. Spiętrzysz pamięć krótką i długą w pełny cykl `sesja -> zapis -> nowa sesja`.
3. Porównasz w notebooku **agenta z pamięcią** i **chatbota bez pamięci**, ale bez dokładania tego benchmarku do UI.
4. Zobaczysz w backend proof, jak zapis sesji realnie wzbogaca kolejny prompt.
5. Odpalisz mocniejsze `Gradio`, w którym żyją `Semantic`, `Working`, `Progress`, `Core` i `Workflow`.
6. Przetestujesz trzy obietnice MVP: **zmiana semantyki na żywo**, **guardrail Core** i **automatyczne KK pilnujące procesu**.
7. Dowieziesz MVP, które daje efekt wow: **zamykasz sesję, otwierasz nową, a Coach wraca do Ciebie z pamięcią**.


## Mapa finału

```text
                 user_id
                    │
                    ▼
         ┌───────────────────────┐
         │   Memory Store        │
         │  (wiele AgentState)   │
         └──────────┬────────────┘
                    │
                    ▼
         start_session(user_id, topic, phase)
                    │
                    ▼
             run_turn(...)  ← write + select
                    │
                    ▼
             close_session(...)  ← compress + write
                    │
                    ▼
         nowa sesja dla tego samego user_id
                    │
                    ▼
           prompt jest bogatszy niż wcześniej
```

To jest najkrótsza definicja dzisiejszego produktu: **ta sama osoba wraca, a agent nie zaczyna od zera**.

```text
     PAMIĘĆ TO NIE JEDEN WOREK NA TOKENY. TO 7 WARSTW, KTÓRE PROJEKTUJESZ.

   ┌────────────────────────────────────────────────────────────────┐
   │ 🏛️ CORE       kim jest agent, czego mu nie wolno    [stałe]    │
   │ 🧠 SEMANTIC   kim jest user + jak mu idzie          [rośnie]   │
   │ 🎯 WORKFLOW   na jakim etapie procesu jesteśmy      [sterownik]│
   │ ⚡ WORKING    co padło przed chwilą                  [ulotne]   │
   │ 📚 EPISODIC   skondensowana historia sesji          [kondensat]│
   │ ───────────────────────────────────────────────────────────────│
   │ 🛠️ PROCEDURAL umiejętności agenta                   [poza MVP] │
   │ 🔧 RESOURCE   narzędzia i zasoby                    [poza MVP] │
   └────────────────────────────────────────────────────────────────┘

   Nie musisz znać Jinja ani Pydantic. Musisz umieć zdecydować:
   • co trafia do której warstwy,
   • kiedy się to zapisuje (w locie vs po sesji),
   • co model widzi w danym momencie (KK).

   To jest inżynieria kontekstu. Kod to tylko dłuto.
```


## Dlaczego dzis `app.py`, a nie inline `Gradio`?

1. To jest **ten sam wzorzec**, ktorego uzywamy w `Agentic AI`.
2. `app.py` jest juz gotowym plikiem w obrazie, wiec notebook nie musi go za kazdym razem przepisywac.
3. Taki uklad jest stabilniejszy w srodowisku JupyterHub niz renderowanie UI bezposrednio w komorce.
4. Unika typowego problemu z `uvicorn.run()` i aktywnym event loopem notebooka.
5. Daje bardziej "prawdziwe" doswiadczenie produktu: masz realna apke na porcie, a nie obiekt w komorce.
6. Rozdziela odpowiedzialnosci: notebook jest przewodnikiem, a `app.py` jest interfejsem.
7. Ulatwia dalszy rozwoj: pozniej mozesz to wyniesc poza notebook praktycznie bez zmiany logiki.


In [ ]:
import difflib
import helper as h

## Od jednego `state` do wielu użytkowników

W Dniu 2 mieliśmy jeden obiekt `state` - pamięć jednej osoby. Prawdziwy produkt działa inaczej: obsługuje wielu ludzi naraz i każdy musi mieć **własny, odseparowany świat pamięci**. Marek nie może zobaczyć błędów Ani, a Ania nie może odziedziczyć celu Marka.

Do tego służy **Memory Store** - tutaj to prosty słownik `user_id -> AgentState`. Na produkcji będzie to baza danych. Dziś zostajemy przy pamięci RAM, bo zgodnie z logiką całego DWthonu **najpierw projektujemy strukturę i zachowanie, dopiero potem storage**.

```text
              MEMORY STORE  (produkcyjnie: baza danych, 1 wiersz = 1 user)
        ┌───────────────────────────────────────────────────────┐
        │  user_id ─────────────────────► AgentState            │
        │                                                       │
        │  "marek_01" ─► [Core|Semantic|Progress|Working|...]   │ ← świat Marka
        │  "ania_02"  ─► [Core|Semantic|Progress|Working|...]   │ ← świat Ani
        └───────────────────────────────────────────────────────┘
                    │                              │
                    ▼                              ▼
             sesja Marka                     sesja Ani
        (jego kot Luna,                  (jej rozmowa o pracę,
         prezentacja NYC)                 jej cele w Londynie)

   ISOLATE na poziomie użytkownika:
   pamięć Marka NIGDY nie wycieka do Ani. To ta sama zasada, co szuflady z Dnia 2 -
   tylko piętro wyżej.
```

Seedujemy **dwie** persony celowo. To nie jest ozdobnik. To najprostszy możliwy dowód operacji `Isolate` z Dnia 2, tylko teraz na poziomie całego produktu.


In [ ]:
store = h.seed_demo_store()
print(h.render_store_overview(store))

## Dlaczego profil semantyczny urósł względem Dnia 2

Wczoraj `StudentProfile` był minimalny, bo ćwiczyliśmy **mechanikę**. Dziś pokazujemy wersję produktową: `Semantic` to nie jedna notatka, tylko zestaw aktywów, które realnie podnoszą jakość kolejnej rozmowy.

Marek nie jest tutaj "nowym userem". To wracający użytkownik z realną stawką biznesową, blokadami pewności siebie, tonem, którego szuka, i kontekstem osobistym. Dlatego jego prompt może być bogaty od pierwszej sekundy.

Nie musisz znać każdego pola z `helper.py`. Zapamiętaj tylko jedno: to wciąż ta sama warstwa `Semantic` z Dnia 2, tylko rozwinięta do formy, którą dałoby się pokazać klientowi albo inwestorowi bez wstydu.


## Żywa pamięć w trakcie rozmowy

W Dniu 2 długa pamięć aktualizowała się głównie **po zamknięciu sesji**. Dziś robimy krok dalej: model przy każdej turze nie tylko odpowiada, ale też proponuje, **co warto dopisać do `Semantic` już teraz**.

Poniżej używamy **kopii** stanu Marka, żeby demo było deterministyczne i nie mieszało późniejszego backend proofu. Chcemy zobaczyć mechanikę, a nie przypadkowy efekt uboczny wcześniejszych uruchomień.

```text
   Uczeń: "Tomorrow my daughter Zosia joins the rehearsal,
           so I want to sound natural and warm."
                          │
                          ▼
                 run_turn_live(state, msg)     ← jedno wywołanie modelu
                          │
          ┌───────────────┴────────────────────┐
          ▼                                     ▼
     reply (do ucznia)               semantic_write (do PAMIĘCI!)
     "Great, Marek..."               personal_facts:  ["córka Zosia"]
                                     upcoming_events: ["rehearsal jutro"]
                                     tone_targets:    ["natural", "warm"]
                          │
                          ▼
        apply_semantic_write(state, ...)  → Semantic rośnie NA ŻYWO,
                                            w środku rozmowy, bez zamykania sesji
```

To jest sedno "żywego organizmu": pamięć semantyczna może rosnąć **w trakcie** rozmowy, a nie dopiero po sesji.

```text
                DWA TEMPA ZAPISU DO PAMIĘCI DŁUGIEJ

   SZYBKIE  (w trakcie rozmowy)         WOLNE  (na koniec sesji)
   run_turn_live()                      close_session()
   ──────────────────────────          ──────────────────────────
   każda tura → semantic_write          cała rozmowa → 1 epizod
   dopisuje świeże fakty OD RAZU        destyluje błędy + next step
   do 🧠 Semantic                       do 📚 Episodic + 📈 Progress

   np. "córka Zosia", ton: "warm"      np. "błąd: want say→want to say,
                                             next: przyimki"

        🧠 rośnie płynnie w tle          📚 porządkuje i kondensuje
```


In [ ]:
demo_state = store.get("marek_01").model_copy(deep=True)
h.start_session(
    demo_state,
    topic="Opening a sales presentation",
    phase=h.SessionPhase.PRACTICE,
)

result = h.run_turn_live(
    demo_state,
    "Tomorrow my daughter Zosia joins the rehearsal, so I want to sound natural and warm.",
)

print("🤖 COACH:")
print(result.reply)
print("\n🧠 CO MODEL POSTANOWIŁ DOPISAĆ DO SEMANTIC:")
print(result.semantic_write.model_dump_json(indent=2, exclude_defaults=True))
print("\n🗂️ FRAGMENT STANU PO TEJ TURZE:")
print({
    "personal_facts": demo_state.profile.personal_context.personal_facts,
    "upcoming_events": demo_state.profile.personal_context.upcoming_events,
    "tone_targets": demo_state.profile.performance_profile.tone_targets,
    "context_notes": demo_state.profile.context_notes,
})
print("\n🛡️ Guardrail Core użyty?", result.core_guardrail_used, "→", result.core_guardrail_reason)


## Ten sam komunikat, dwa światy

UI nie potrzebuje już osobnego benchmarku, ale sama idea jest nadal bardzo cenna dydaktycznie. Dlatego porównanie "z pamięcią" vs "bez pamięci" robimy tutaj, w kontrolowanej komórce, gdzie timing jest jasny i nic nie myli użytkownika.


In [ ]:
# Benchmark: ta sama wiadomość, dwa światy
msg = "The CFO will be in the room tomorrow, so I need to sound calm and executive."

blank = h.blank_memory_clone("marek_01", "Opening a sales presentation")
h.start_session(blank, topic="Opening a sales presentation", phase=h.SessionPhase.PRACTICE)
print("🟥 BEZ PAMIĘCI:")
print()
print(h.run_turn(blank, msg))

full = store.get("marek_01").model_copy(deep=True)
h.start_session(full, topic="Opening a sales presentation", phase=h.SessionPhase.PRACTICE)
print()
print("🟩 Z PAMIĘCIĄ:")
print()
print(h.run_turn(full, msg))


## Szybki dowód backendu

Teraz robimy już pełny mini-cykl pamięci, ale nadal bez UI. Chodzi o to, żeby zobaczyć dokładnie te same operacje, które wczoraj rozkładaliśmy na części:

1. Otwieramy sesję i ustawiamy fazę procesu.
2. Robimy pierwszy `Select`, czyli budujemy prompt przed rozmową.
3. Wysyłamy turę użytkownika i wykonujemy `Write` do pamięci roboczej.
4. Dostajemy odpowiedź Coacha i zamykamy turę po stronie modelu.
5. Przy `close_session(...)` uruchamiamy `Compress`, czyli destylację sesji do epizodu.
6. Zapisujemy epizod oraz progres z powrotem do stanu.
7. Budujemy nowy prompt i patrzymy w `diff`, czy system naprawdę stał się bogatszy.

```text
   SESJA 1 (poniedziałek)                 SESJA 2 (środa)
   ─────────────────────                  ─────────────────────
   Coach: "Cześć Marek! Jak Luna?"        Coach: "Wracamy do przyimków
   Uczeń: "I want say welcome..."                 i want to + verb..."
             │                                       ▲
             ▼                                       │
      [⚡ Working rośnie]                             │  🧠 PAMIĘĆ
             │                                       │  PRZYWOŁANA
     [Zamknij i zapisz]                              │
             │                                       │
             ▼                                       │
   ┌───────────────────────────┐                     │
   │ 📚 Episodic:  +1 epizod   │─────────────────────┘
   │ 📈 Progress:  błędy+focus │  te same dane wchodzą
   │ 🧠 Semantic:  +fakty      │  do promptu następnej sesji
   └───────────────────────────┘

   Między sesjami Working znika, ale Episodic/Progress/Semantic zostają.
   Dlatego środa nie zaczyna od zera.
```


In [ ]:
state = store.get("marek_01").model_copy(deep=True)
h.start_session(
    state,
    topic="Opening a sales presentation",
    phase=h.SessionPhase.PRACTICE,
)

before_prompt = h.preview_next_prompt(
    state,
    topic="Opening a sales presentation",
    phase=h.SessionPhase.PRACTICE,
)

reply = h.run_turn(
    state,
    "Tomorrow Sarah from Boston joins my demo. I want say: welcome everybody on the meeting.",
)

episode = h.close_session(state)

after_prompt = h.preview_next_prompt(
    state,
    topic="Opening a sales presentation",
    phase=h.SessionPhase.PRACTICE,
)

diff = "\n".join(
    difflib.unified_diff(
        before_prompt.splitlines(),
        after_prompt.splitlines(),
        fromfile="before_save",
        tofile="after_save",
        lineterm="",
    )
)

print("🤖 COACH:")
print(reply)
print("\n📚 EPIZOD PO ZAPISIE:")
print(episode.model_dump_json(indent=2))
print("\n🧠 STAN PO ZAPISIE:")
print({
    "recent_mistakes": state.progress.recent_mistakes,
    "next_focus": state.progress.next_focus,
    "homework_status": state.progress.homework_status,
    "episodes": len(state.episodic.episodes),
    "semantic_summary": state.profile.context_notes,
})
print("\n🧬 PROMPT DIFF PO ZAPISIE:")
print(diff or "Brak różnic")


W tym miejscu powinno być już jasne, że sam silnik dowozi MVP: po zapisie sesji coś naprawdę wraca do promptu. Teraz przechodzimy poziom wyżej i pakujemy ten mechanizm w produkt, który daje efekt wow w przeglądarce.


## Co siedzi w `app.py`

Od tego momentu `app.py` traktujemy jak **gotowy plik produktu**, a nie kod do przepisywania w notebooku. Notebook jest orkiestratorem i przewodnikiem. `helper.py` niesie logikę pamięci. `app.py` tylko pokazuje ją jako interaktywny produkt.

```text
   NOTEBOOK (day3)          helper.py                app.py
   ---------------          ---------                ------
   ORKIESTRATOR       ->    SILNIK PAMIECI     ->    INTERFEJS (Gradio)
   "co uruchamiamy"        "jak działa pamięć"      "jak to wygląda i klika"

   - backend proof          - AgentState + 7 warstw   - panele, przyciski, CSS
   - odpala app.py          - run_turn / run_turn_live- na klik woła funkcje
   - linki, instrukcje      - compress / close_session  z helper.py
                            - seed_demo_store()       - nie zna logiki pamięci

   Zasada: logika pamięci mieszka w helperze. app.py tylko ją POKAZUJE.
```

```text
   KLIK "Wyślij" w przeglądarce
        |
        v
   handler w app.py: send_message_ui(user_id, msg)
        |  1. weź AgentState ze Store po user_id
        |  2. zrób SNAPSHOT "przed"
        v
   helper.run_turn_live(state, msg)   <- cała pamięć dzieje się TU
        |  model odpowiada + proponuje semantic_write
        v
   SNAPSHOT "po" -> compare_snapshots() -> które warstwy się zmieniły
        |
        v
   render_all(user_id) -> przerysuj 16 paneli naraz
        |
        v
   UI ożywa: puls mruga - Semantic dostaje chip NEW - Heartbeat dopisuje ślad
```

```text
   app.py to OSOBNY proces (port 8080) z WLASNYM Memory Store w RAM.
   ┌──────────────┐        ┌──────────────────────────┐
   │  notebook    │  ---   │  proces app.py (8080)    │
   │  (swój store)│  brak  │  seed_demo_store() w RAM │
   └──────────────┘  wspól.└──────────────────────────┘

   - Restart komórki `!python app.py` = czysty start pamięci demo.
   - `Reset usera` w UI cofa jedną personę do seeda, bez restartu procesu.
```


## Co ma ten UI, czego nie miała poprzednia wersja?

1. Czytelny font i kontrast, który nie psuje polskich znaków ani hierarchii treści.
2. Jasny flow `co zrobić teraz`, widoczny od pierwszej sekundy bez tłumaczenia ustnego.
3. Pasek pulsu warstw wysoko na ekranie, więc od razu widać, która pamięć właśnie zareagowała.
4. Widoczny panel `co się właśnie zmieniło` oraz `Heartbeat systemu`, zamiast chowania kluczowej delty w debugowym akordeonie.
5. `Semantic`, `Working`, `Progress`, `Core`, `Workflow` i `Episodic` pokazane jako **żywy organizm**, a nie jako jeden blob stanu.
6. `KK` działające automatycznie w tle, bez zmuszania użytkownika do ręcznego przełączania trybów.
7. Warstwa debug dla ambitnych pod spodem: `Prompt Preview`, `Last Session Report`, `Store Overview` i export JSON.


## 🆘 Zanim odpalisz serwer

1. Uruchom komórkę `!python memory_tutor_system/app.py` i nie zamykaj jej od razu.
2. Samo `[*]` przy komórce **nie wystarcza** - to tylko znak, że proces jeszcze działa.
3. Wejdź do UI dopiero wtedy, gdy w logu zobaczysz **`Application startup complete.`** oraz **`Uvicorn running on http://0.0.0.0:8080`**.
4. Jeśli klikniesz za wcześnie, możesz mieć wrażenie, że link nie działa, mimo że serwer jeszcze tylko się podnosi.
5. Jeśli zobaczysz `Address already in use`, zatrzymaj starą komórkę `Stop ⏹️` i uruchom serwer jeszcze raz.
6. W polu wiadomości działa skrót: **`Enter = wyślij`**, **`Shift+Enter = nowa linia`**.
7. Jeśli UI w ogóle się nie otwiera, najpierw sprawdź log startowy i `proxy/8080`, a dopiero potem pytaj na Slacku.


In [ ]:
!python app.py

## Otwórz UI w nowej karcie

Gdy komórka powyżej zacznie "mielić", **nie klikaj od razu**. Najpierw poczekaj, aż w logu zobaczysz taki wzorzec:

To normalne, że komórka z `!python memory_tutor_system/app.py` będzie miała `[*]`. Aplikacja działa wtedy jako osobny proces. Ale samo `[*]` nie oznacza jeszcze, że serwer już jest gotowy - gotowość potwierdzają dopiero logi powyżej.


![](../images/gradio_app.jpg)

Dopiero wtedy otwieraj link:

## 👉👉 [Klik => MEMORY UI (Port 8080)](/hub/user-redirect/proxy/8080/) 👈👈


## Jak czytać ten ekran

1. 🟢 **Puls warstw** na samej górze pokazuje 7 kart pamięci; po każdej akcji karta, która się zmieniła, podświetla się na zielono.
2. 💬 **Rozmowa** po lewej to miejsce, gdzie testujesz produkt jak zwykły użytkownik - bez ręcznego grzebania w stanie.
3. ❤️ **Prawa kolumna** z personą, panelem `🔴 Co się właśnie zmieniło` i `💓 Heartbeat` tłumaczy po ludzku, co system zrobił z Twoją wiadomością.
4. 🧠 **Sekcja niżej** z `Semantic`, `Working` i `Progress` pokazuje szczegóły tego, co puls zasygnalizował u góry.
5. 🔧 **`Pod maską`** zostawiliśmy dla ambitnych - tam są `Core`, `Workflow`, `Episodic`, `Prompt Preview` i `Store Overview`.
6. ✍️ **Scenariusze** `🗣️🎯👤🛡️🔁` tylko wpisują gotowe zdanie do pola - żeby ruszyć turę, kliknij `Wyślij` albo użyj `Enter`.
7. ⭐ **Złota ścieżka do wow** to: `⚡ Marek -> Otwórz sesję -> 👤 Ludzki detal -> Wyślij -> 🛡️ Granica Core -> Wyślij -> Zamknij i zapisz -> Otwórz sesję -> ⚡ Ania`.

![](../images/gradio_1.jpg)

## Scenariusz demo wow

1. Kliknij **`⚡ Marek: Sales demo`**.
2. Kliknij **`Otwórz sesję`** i zobacz pierwszy komunikat Coacha.
3. Kliknij **`🎯 Wysoka stawka`** albo **`👤 Ludzki detal`**, a potem **`Wyślij`** i sprawdź, że `Semantic` zmienia się **na żywo**.
4. Kliknij **`🗣️ Błąd językowy`**, a potem **`Wyślij`** i zobacz, jak rośnie `Working`, a system łapie słabości i vocabulary.
5. Kliknij **`🛡️ Granica Core`**, a potem **`Wyślij`** i sprawdź, że guardrail widać w **`🔴 Co się właśnie zmieniło`** oraz na karcie **`Core`** w pulsie warstw.
6. Kliknij **`Zamknij i zapisz`** i zobacz, że budzi się `Episodic` oraz `Last Session Report`.
7. Kliknij **`Otwórz sesję`** jeszcze raz, wyślij `Let's continue.`, a na końcu przełącz się na Anię i potwierdź izolację pamięci.


![](../images/gradio_2.jpg)

## Self-check MVP

1. Czy `Working Memory` rośnie od pierwszej wiadomości i pokazuje realne okno modelu?
2. Czy po przycisku **`Wysoka stawka`** albo **`Ludzki detal`** widać natychmiast zmianę w `Semantic Memory`?
3. Czy po przycisku **`Granica Core`** guardrail widać w **`🔴 Co się właśnie zmieniło`** oraz na karcie **`Core`** w pulsie warstw?
4. Czy `Workflow + KK` zmienia fazę **automatycznie**, bez ręcznego przełączania przez użytkownika?
5. Czy po kliknięciu **`Zamknij i zapisz`** liczba epizodów rośnie i pojawia się `Last Session Report`?
6. Czy po ponownym `Otwórz sesję` widać **`PAMIĘĆ PRZYWOŁANA`**, a Coach nie wraca do zera?
7. Czy po przełączeniu z `Marek` na `Ania` widać, że to dwa odseparowane światy pamięci?


<div class="alert alert-block alert-danger">

<h2>✅ Zaliczenie Dnia 3</h2>
<h4>Co masz teraz zrobić?</h4>

<ol>
    <li>📸 Zrób screenshot głównego widoku z widoczną personą i pulsującymi warstwami pamięci.</li>
    <li>📸 Zrób drugi screenshot po teście <code>Wysoka stawka</code> albo <code>Ludzki detal</code>, tak aby było widać zmianę w <code>Semantic Memory</code>.</li>
    <li>📸 Zrób trzeci screenshot po teście <code>Granica Core</code>, tak aby było widać guardrail.</li>
    <li>📸 Zrób czwarty screenshot po <code>Zamknij i zapisz</code>, tak aby było widać <code>Episodic</code> albo <code>Last Session Report</code>.</li>
    <li>🔁 Przełącz z <code>Marek</code> na <code>Ania</code> i potwierdź izolację pamięci między personami.</li>
    <li>🧪 Zrób własne eksperymenty – pamiętaj, że to żywy organizm, który trzeba kalibrować i stabilizować na realnych przykładach.</li>
    <li>💬 Wrzuć materiał na Slacka na kanał <code><a href="https://dataworkshopclub.slack.com/archives/C0AD0KN2SMR">#dwthon_memory_day3_done</a></code> i dopisz jedno zdanie: który moment najmocniej pokazał Ci, że to już jest system pamięci, a nie zwykły chat?</li>
</ol>
</div>

### 🎁 Punkty

Za domknięcie Dnia 3 wpada ostatnia paczka punktów bonusowych i komplet do certyfikatu.


## Podsumowanie Dnia 3

1. Przeszliśmy od jednego `state` do **produktu obsługującego wielu użytkowników z izolacją pamięci**.
2. Pokazaliśmy, że `Semantic` może rosnąć **na żywo w trakcie rozmowy**, a nie tylko po sesji.
3. Domknęliśmy pełny cykl `sesja -> zapis -> nowa sesja`, więc pamięć naprawdę wraca do kolejnego promptu.
4. Dowieźliśmy publiczną obietnicę: Coach pamięta **użytkownika, kontekst i postęp**.
5. Pokazaliśmy, że `Core` i `KK` pilnują jakości produktu w tle, bez ręcznego sterowania przez użytkownika.
6. Opakowaliśmy wszystko w `Gradio`, które wygląda jak wartościowe MVP, a nie jak surowy panel techniczny.
7. Zostawiliśmy naturalny most do pełnej ścieżki `Agentic AI`, bo potencjał jest już wyraźny, ale widać też, co będzie skalowane dalej.


## Co dalej?

DWthon Memory to **trzeci z pięciu DWthonów**. Każdy z nich pokazuje wycinek większej całości: pozwala szybko wejść w temat, zbudować coś działającego i sprawdzić, czy taki sposób pracy z AI jest tym, co chcesz rozwijać dalej.

Choć dzisiejsza edycja dobiega końca, Twoja nauka wcale nie musi. Największe postępy zwykle pojawiają się wtedy, gdy można wrócić do materiału po kilku dniach, jeszcze raz przejść trudniejszy fragment i spokojnie wdrożyć pomysły we własnym projekcie. Tę swobodę daje pełna ścieżka.


<div style="text-align: center;">
  <a href="https://pay.dataworkshop.eu/?cart=%7B%22products%22%3A%7B%22212%22%3A1%7D%2C%22discountCode%22%3A%22season_pass_founding%22%7D&lang=pl&currency=pln&gate=pl"><img src="../images/seasson_pass.jpg" width="900" alt="Seasson Pass"></a>
</div>



Cały program **Software 3.0** składa się z dwóch kursów: **Praktyczny LLM** oraz **Agentic AI**. DWthony są intensywnym wprowadzeniem. Pełne kursy pozwalają nie tylko zobaczyć rozwiązania, ale przede wszystkim zrozumieć, dlaczego działają i samodzielnie wykorzystać je w praktyce. Jeżeli po tych trzech dniach czujesz, że chcesz lepiej zrozumieć fundamenty pracy z modelami językowymi — od structured output, przez reprezentację wiedzy i retrieval, aż po zarządzanie kontekstem i projektowanie architektury aplikacji — kolejnym krokiem będzie **Praktyczny LLM**. Dzięki temu kolejne projekty będzie Ci się budować szybciej, pewniej i z dużo większą świadomością. Jeżeli z kolei najbardziej zainteresował Cię temat pamięci agentów i chcesz pójść znacznie dalej — w persistence, `skills`, `resource memory`, pracę z narzędziami, monitoring czy architekturę systemów wieloagentowych — tę część rozwijamy w **Agentic AI**. Tam budujemy rozwiązania bliższe temu, jak projektuje się systemy działające w środowisku produkcyjnym.

Jaki powinien być Twój kolejny krok w ścieżce Software 3.0, odpowiedź jest prosta:

- [Season Pass / cała ścieżka 5 DWthonów](https://pay.dataworkshop.eu/?cart=%7B%22products%22%3A%7B%22212%22%3A1%7D%2C%22discountCode%22%3A%22season_pass_founding%22%7D&lang=pl&currency=pln&gate=pl) - jeśli chcesz mieć pełny przegląd serii i uczyć się w swoim rytmie.
- [Praktyczny LLM](https://dataworkshop.eu/pl/praktyczny-llm) - jeśli chcesz uporządkować fundamenty i przerabiać materiał spokojnie, po swojemu.
- [Agentic AI](https://dataworkshop.eu/agentic-ai) - jeśli chcesz rozwinąć pamięć w pełny system i zobaczyć, jak takie rozwiązania składa się na poziomie produkcyjnym.

## Na koniec: zapraszam na <a href="https://www.youtube.com/watch?v=jNOOElDKcVI">sesje Q&A</a>

Jeśli chcesz spokojnie wrócić do pytań, które padły po warsztacie, obejrzyj zapis Q&A. To dobra opcja, gdy chcesz jeszcze raz poukładać decyzje architektoniczne, edge case'y i różnicę między MVP a pełną ścieżką - już bez presji czasu.


<div style="text-align: center;">
  <a href="https://www.youtube.com/watch?v=jNOOElDKcVI"><img src="../images/sesja_qa.jpg" width="1000" alt="Seasson Pass"></a>
</div>

